In [1]:
# directory check and change
import os

current_dir = os.getcwd()

if current_dir == '/mnt/datavisor':
    os.chdir("Elina_Proj/Galileo_SML")
    print(f"Changed directory to: {os.getcwd()}")
elif os.path.basename(current_dir) == "Galileo_SML":
    print("This notebook is in the correct directory")
else:
    print(f"Current working directory is: {current_dir}")

Changed directory to: /mnt/datavisor/Elina_Proj/Galileo_SML


In [4]:
import os, csv, math, json, logging, importlib
import matplotlib.pyplot as plt
import pandas as pd
import dask.dataframe as dd
import util.testset_evaluation as tse

from pydatatools import (
    data_sync, data_preprocessor, feature_selection, data_package,
    model_train, model_scorecard, model_evaluate, model_onboarding,
)
from pydatatools.util import label_util, plot_util

from dask_gateway import Gateway
import dask, distributed

dask.config.set({"distributed.comm.timeouts.tcp": "100s"})
logger = logging.getLogger(__name__)

from bokeh.io import output_notebook
output_notebook()

Loading BokehJS ...

In [5]:
gateway = Gateway()
clusters = gateway.list_clusters()
# clusters
cluster = gateway.connect(clusters[-1].name)
dask_client = cluster.get_client()
dask_client

Connection method: Cluster object,Cluster type: dask_gateway.GatewayCluster
Dashboard: /services/dask-gateway/clusters/trialtools.40b29bd737134caab87a5c443f67596a/status,


In [4]:
import pandas as pd
import dask.dataframe as dd
import os

# Read the user list CSV and deduplicate user_id
user_list_path = 'Data/sofi_uml_userIdList_20250701-20250714.csv'
path = 's3://datavisor-trial-sofipoc-awsuswest2preprod/DATASET/20250701_20250714/IP_HISTORY/'

user_list_df = pd.read_csv(user_list_path)
user_id_list = user_list_df['user_id'].drop_duplicates().tolist()

# Print info about user list
print(f"[INFO] Loaded {len(user_list_df)} user ids from {user_list_path}, {len(user_id_list)} unique.")

# List all CSV files in the S3 path
from fsspec.core import url_to_fs

fs, _ = url_to_fs(path)
csv_files = [f's3://{f}' if not str(f).startswith('s3://') else str(f)
             for f in fs.glob(os.path.join(path, '*.csv'))]

print(f"[INFO] Found {len(csv_files)} CSV files in {path}")

output_path = 's3://datavisor-trial-sofipoc-awsuswest2preprod/DATASET/20250701_20250714/IP_HISTORY_FILTERED/'

for idx, csv_file in enumerate(csv_files):
    print(f"[INFO] Processing file {idx+1}/{len(csv_files)}: {csv_file}")
    # Read each file as a Dask DataFrame
    ddf = dd.read_csv(csv_file)
    # Filter rows by user_id
    filtered = ddf[ddf['user_id'].isin(user_id_list)]
    # Pick a file name based on the input
    base_name = os.path.basename(csv_file)
    output_file = os.path.join(output_path, f'filtered_{base_name}')
    # Write filtered data to a single CSV file
    filtered.to_csv(output_file, single_file=True, index=False)
    # Log info about filtering
    total_rows = ddf.shape[0].compute()
    filtered_rows = filtered.shape[0].compute()
    print(f"[INFO] {csv_file}: {total_rows} total rows, {filtered_rows} rows matched and written to {output_file}")

[INFO] Loaded 169822 user ids from Data/sofi_uml_userIdList_20250701-20250714.csv, 169822 unique.
[INFO] Found 14 CSV files in s3://datavisor-trial-sofipoc-awsuswest2preprod/DATASET/20250701_20250714/IP_HISTORY/
[INFO] Processing file 1/14: s3://datavisor-trial-sofipoc-awsuswest2preprod/DATASET/20250701_20250714/IP_HISTORY/sofi_uml_20250701_IP_HISTORY.csv
[INFO] s3://datavisor-trial-sofipoc-awsuswest2preprod/DATASET/20250701_20250714/IP_HISTORY/sofi_uml_20250701_IP_HISTORY.csv: 2833474 total rows, 309730 rows matched and written to s3://datavisor-trial-sofipoc-awsuswest2preprod/DATASET/20250701_20250714/IP_HISTORY_FILTERED/filtered_sofi_uml_20250701_IP_HISTORY.csv
[INFO] Processing file 2/14: s3://datavisor-trial-sofipoc-awsuswest2preprod/DATASET/20250701_20250714/IP_HISTORY/sofi_uml_20250702_IP_HISTORY.csv
[INFO] s3://datavisor-trial-sofipoc-awsuswest2preprod/DATASET/20250701_20250714/IP_HISTORY/sofi_uml_20250702_IP_HISTORY.csv: 2882025 total rows, 317163 rows matched and written to s

In [5]:
from fsspec.core import url_to_fs

# List all CSV files in the filtered S3 path
filtered_fs, _ = url_to_fs(output_path)
filtered_csv_files = [
    f's3://{f}' if not str(f).startswith('s3://') else str(f)
    for f in filtered_fs.glob(os.path.join(output_path, '*.csv'))
]

print(f"[INFO] Found {len(filtered_csv_files)} filtered CSV files to drop 'time' column.")

for idx, csv_file in enumerate(filtered_csv_files):
    print(f"[INFO] Dropping 'time' column in file {idx+1}/{len(filtered_csv_files)}: {csv_file}")
    # Read into Dask DataFrame
    ddf = dd.read_csv(csv_file)
    # Drop 'time' column if it exists
    if 'time' in ddf.columns:
        ddf = ddf.drop('time', axis=1)
        # Overwrite the file at the original path
        ddf.to_csv(csv_file, single_file=True, index=False)
        print(f"[INFO] Dropped and saved without 'time': {csv_file}")
    else:
        print(f"[INFO] No 'time' column found in: {csv_file}")


[INFO] Found 14 filtered CSV files to drop 'time' column.
[INFO] Dropping 'time' column in file 1/14: s3://datavisor-trial-sofipoc-awsuswest2preprod/DATASET/20250701_20250714/IP_HISTORY_FILTERED/filtered_sofi_uml_20250701_IP_HISTORY.csv
[INFO] Dropped and saved without 'time': s3://datavisor-trial-sofipoc-awsuswest2preprod/DATASET/20250701_20250714/IP_HISTORY_FILTERED/filtered_sofi_uml_20250701_IP_HISTORY.csv
[INFO] Dropping 'time' column in file 2/14: s3://datavisor-trial-sofipoc-awsuswest2preprod/DATASET/20250701_20250714/IP_HISTORY_FILTERED/filtered_sofi_uml_20250702_IP_HISTORY.csv
[INFO] Dropped and saved without 'time': s3://datavisor-trial-sofipoc-awsuswest2preprod/DATASET/20250701_20250714/IP_HISTORY_FILTERED/filtered_sofi_uml_20250702_IP_HISTORY.csv
[INFO] Dropping 'time' column in file 3/14: s3://datavisor-trial-sofipoc-awsuswest2preprod/DATASET/20250701_20250714/IP_HISTORY_FILTERED/filtered_sofi_uml_20250703_IP_HISTORY.csv
[INFO] Dropped and saved without 'time': s3://datavis